<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 10 · Bond Portfolio Construction

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch10_bond_portfolio_construction.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

A bond portfolio is an aggregated cash-flow and curve-risk profile, not a list of holdings. In this lab you aggregate market value, duration, and DV01 for a three-bond book, compare bullet, barbell, and ladder allocations whose headline durations sit close together, and solve the market-value weights that hit a DV01 risk budget. You finish able to explain why duration is a value-weighted average while DV01 adds in currency units, and why similar durations can hide very different exposures across the curve.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The script aggregates the 10 million USD three-bond book into a value-weighted duration of 4.95 years and a total DV01 of 4,950 USD per basis point. It then rebuilds the book as bullet, barbell, and ladder allocations, printing durations of 5.08, 5.36, and 5.26 years with matching DV01s, and closes with each bond's share of total portfolio DV01.

In [ ]:
SCRIPT = ROOT / "code" / "ch10_bond_portfolio_construction.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

1. Verify the aggregation: value weights of 40/30/30 on durations of 2.1, 5.0, and 8.7 years give 4.95, and the DV01s 840, 1,500, and 2,610 sum to 4,950 USD per basis point.
2. Confirm the three profiles sit within about 0.3 duration years of each other yet spread value very differently across the curve, which is exactly why one headline duration is not enough.
3. Check that the long bond contributes 52.73% of total DV01 on only 30% of market value, and explain the leverage that duration adds to the risk budget.
4. Note the units: durations in years and DV01 in USD per basis point, computed as market value times duration divided by 10,000.
5. Treat the bond and profile tables as illustrative teaching inputs.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch10_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Hit a DV01 risk budget with value weights

Because DV01 adds in currency units, a risk budget is a set of target DV01 shares, not market-value shares. This lab solves the value weights that make three illustrative bonds contribute 20%, 30%, and 50% of portfolio DV01, given each bond's DV01 per unit of market value. The dv01_share column is the acceptance test.

In [ ]:
import pandas as pd

durations = pd.Series({"short": 2.1, "belly": 5.0, "long": 8.7})
budgets = pd.Series({"short": 0.20, "belly": 0.30, "long": 0.50})
raw = budgets / durations  # budget relative to duration
weights = raw / raw.sum()  # market-value weights that hit the budget
shares = weights * durations  # DV01 contributions
summary = pd.DataFrame({
    "weight": weights.round(4),
    "dv01_share": (shares / shares.sum()).round(4),
})
print(summary)
print("portfolio duration:", round(shares.sum(), 4))

### What to observe

The value weights are 44.77% short, 28.21% belly, and 27.02% long, yet the achieved DV01 shares are exactly 20%, 30%, and 50%. The portfolio duration is 4.7013 years, so a 10 million USD book run this way carries about 4,701 USD per basis point, split roughly 940 / 1,410 / 2,351 USD across the three bonds.

## Exercises

1. **Reconcile the risk budget.** Confirm the printed DV01 shares of 0.1697, 0.3030, and 0.5273 divide each bond's DV01 by the 4,950 total. *Hint:* the shares must sum to one.
2. **Rebuild the profile durations.** Recompute 5.08, 5.36, and 5.26 from the profile weights and the three bond durations. *Hint:* each profile duration is a value-weighted average.
3. **Hit a duration target.** Find the short/belly/long mix that reaches a 6.0 duration with no bond above 60% of value. *Hint:* solve the linear constraints by hand before coding them.
4. **Size the scenario loss.** Apply a 25-basis-point parallel rise to each profile's DV01 and rank the three losses. *Hint:* the loss in USD is DV01 times 25.
5. **Break the budget.** Change the mini-lab budgets to equal thirds and report the new weights and the resulting duration. *Hint:* equal DV01 shares pull the weights toward the short bond.

For more detail, see Chapter 10 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">